# Held-out synthetic test sets (Task 2 + Task 3)

**Purpose.** The two real role-play recordings (consult A and B) were used to choose models, read errors and fix the role
rules, so they are *development* data. This notebook builds **new test data that no system has seen** and scores the
**frozen** systems on it exactly once.

| Test set | Content | Measures |
|---|---|---|
| Clinical-vocabulary sentences | 100 sentences × 6 voices = 600 clips | ASR WER / CER and **drug / dose / test-name accuracy** (Task 2, Task 3C) |
| Synthetic consultations | 6 conversations, 2–5 speakers, overlaps, one with fan noise | DER, role error, cpWER / cpCER (Task 3, Task 1 improvement) |

**Speech is generated** with AI4Bharat **Indic Parler-TTS** (Apache 2.0, 6 Bengali voices). The texts
(`data/tts/*.tsv`) were written before any system output on them was seen (see the order of entries in `notes.md`).

**Caveats (reported with the results).**
- TTS speech is cleaner and more regular than real speech, so scores are optimistic; real held-out recordings remain the stronger test.
- The scripts were written by the same person who wrote the role rules.
- The TTS can mispronounce words: 60 random clips are listed for a check by ear (`data/tts/listen_check.csv`, `ok = y/n`). **This check was not done for the submission**, so no clip was excluded and the reported scores use all 600 clips.

**Runtime.** Colab, T4 GPU. About 70 min, most of it TTS generation (≈ 5 s per clip).

## 1. Setup

Requirements before running:
1. Runtime → Change runtime type → **T4 GPU**.
2. A Hugging Face token in Colab **Secrets** as `HF_TOKEN` (notebook access on). The token is never printed or written to disk.
3. Accept the terms of the gated models with that account:
   [indic-parler-tts](https://huggingface.co/ai4bharat/indic-parler-tts),
   [indic-conformer-600m-multilingual](https://huggingface.co/ai4bharat/indic-conformer-600m-multilingual),
   [pyannote/speaker-diarization-3.1](https://huggingface.co/pyannote/speaker-diarization-3.1),
   [pyannote/segmentation-3.0](https://huggingface.co/pyannote/segmentation-3.0).
The code is cloned from GitHub.

In [ ]:
import os, shutil, subprocess
from google.colab import userdata

REPO = "/content/aimscribe"                               # code + labels
DATA = "/content/aimscribe_data"                          # audio + models (AIMSCRIBE_DATA)
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/sahilaf/Aimscribe", REPO], check=True)
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["AIMSCRIBE_DATA"] = DATA


def sh(cmd):
    # Run a shell command from the repository root, stream its output, stop the notebook on failure.
    p = subprocess.run(cmd, shell=True, executable="/bin/bash", cwd=REPO)
    if p.returncode:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")


sh("nvidia-smi --query-gpu=name,memory.total --format=csv")

## 2. Python environments

Two separate environments, because their pins conflict:

| Environment | Used for | Key versions |
|---|---|---|
| `/content/venv_tts` | speech generation | `parler-tts` (pins transformers 4.46.1), torch 2.5.1 |
| `/content/venv_eval` | the frozen pipeline and scoring | the project's `requirements.txt` + `constraints.txt` (same pins as the laptop: torch 2.5.1, pyannote.audio 3.3.2, transformers 4.46.3, onnxruntime 1.20.1) |

Colab's default Python is 3.13; torch 2.5.1 has no 3.13 wheels, so `uv` installs a managed Python 3.12.

In [ ]:
sh("pip install -q uv && uv python install 3.12")
TORCH = "torch==2.5.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121"
for venv in ["venv_tts", "venv_eval"]:
    sh(f"uv venv -q --python-preference only-managed --python 3.12 /content/{venv}")
    sh(f"uv pip install -q --python /content/{venv}/bin/python {TORCH}")
sh('uv pip install -q --python /content/venv_tts/bin/python "git+https://github.com/huggingface/parler-tts.git" '
   'librosa soundfile silero-vad pandas matplotlib')
sh("uv pip install -q --python /content/venv_eval/bin/python -r requirements.txt -c constraints.txt")
TTS_PY, EVAL_PY = "/content/venv_tts/bin/python", "/content/venv_eval/bin/python"
sh(f"{TTS_PY} -c 'import parler_tts, transformers; print(\"tts env OK\", transformers.__version__)'")
sh(f"{EVAL_PY} -c 'import pyannote.audio, onnxruntime, transformers; print(\"eval env OK\", transformers.__version__)'")

## 3. Generate the test audio

`data/make_tts_tests.py` (see its docstring for details):
- **Consultations:** each line is synthesised in its speaker's voice and placed on a timeline. The `timing` column of
  `consult_scripts.tsv` creates interruptions (`cut:X`) and backchannels (`inside`). Speech regions come from Silero VAD on
  each clean line, so the reference RTTM is exact. S06 gets low-frequency "fan" noise at 10 dB SNR.
- **Sentences:** 100 × 6 voices. Seeds are fixed per clip (reproducible). A clip whose speaking rate is implausible for
  its text (outside 4–25 characters/s) is regenerated with another seed, up to 3 tries.

Sentence generation runs as 3 processes on the same GPU (about 30 % faster than one) and resumes if interrupted.

In [ ]:
sh(f"cd data && {TTS_PY} make_tts_tests.py --part consultations")

In [ ]:
procs = [subprocess.Popen(f"cd data && {TTS_PY} make_tts_tests.py --part sentences --shard {i} --n_shards 3 > /content/tts_shard{i}.log 2>&1",
                          shell=True, executable="/bin/bash", cwd=REPO) for i in range(3)]
codes = [p.wait() for p in procs]
print("shard exit codes:", codes, "| clips:", len([f for f in os.listdir(f"{DATA}/tts_sentences") if f.endswith(".wav")]), "/ 600")
sh(f"cd data && {TTS_PY} make_tts_tests.py --part manifest")   # data/tts/sentences_manifest.csv + listen_check.csv

## 4. Task 3: the full pipeline on the 6 synthetic consultations

Both runs use the same ASR (IndicConformer 600M, RNN-T). They differ only in the Task 1 / Task 2 improvements:

| Run | Diarization | VAD gate |
|---|---|---|
| `tts_baseline` | pyannote 3.1 defaults | off |
| `tts_final` | pyannote 3.1 + segment-level re-clustering (threshold 0.70, tuned in Task 1) | on |
| `tts_final_c1` | pyannote community-1 (the final pipeline; run in the pyannote 4 env, `envs/requirements-pyannote4.txt`) | on |

Metrics (`task3_pipeline/evaluate_pipeline.py`): DER and role error (collar 0.25 s), CER over all text,
role-attributed WER / CER, and speaker-aware **cpWER / cpCER**, which also penalises mixing up two speakers with the same role.

In [ ]:
IC = f"{DATA}/models/indic-conformer-600m"
from huggingface_hub import snapshot_download   # local folder: the ONNX weights refuse the HF cache's symlinks
snapshot_download("ai4bharat/indic-conformer-600m-multilingual", local_dir=IC, token=os.environ["HF_TOKEN"])
for run, flag in [("tts_baseline", "--baseline"), ("tts_final", "--diarizer pyannote31_recluster")]:
    sh(f"{EVAL_PY} task3_pipeline/pipeline.py {flag} --asr_model {IC} --audio {DATA}/tts_consultation/S0*.wav --out_dir results/task3/{run} > /content/{run}.log 2>&1")
    sh(f"{EVAL_PY} task3_pipeline/evaluate_pipeline.py --run {run} --set tts_consultation > /dev/null")

In [ ]:
import json
import pandas as pd

rows = []
for run in ["tts_baseline", "tts_final"]:
    ev = json.load(open(f"{REPO}/results/task3/{run}/eval.json", encoding="utf-8"))
    for rec, v in ev.items():
        rows.append({"run": run, "consultation": rec, **{k: v[k] for k in ["DER%", "role_error_rate%", "CER%_all_text", "cpWER%", "cpCER%"]},
                     "roles": "" if rec == "OVERALL" else ", ".join(sorted(v["predicted_roles"].values()))})
pd.DataFrame(rows).set_index(["run", "consultation"])

## 5. Task 2: clinical-vocabulary sentence test

`task2_asr/eval_tts_sentences.py` scores WER / CER per tag (MED, EN, SHORT) and per voice, plus **term accuracy**: the share
of listed drug / dose / test names that appear in the output. A term counts if it starts a word, so Bengali suffixes may
follow (থাইরয়েড → থাইরয়েডের); a second figure also accepts spacing-only variants (ফলোআপে vs ফলো আপে).
Clips marked `ok = n` in `listen_check.csv` would be excluded; in the submitted sheet none is marked yet, so all clips are scored.

In [ ]:
sh(f"{EVAL_PY} task2_asr/eval_tts_sentences.py --backend indicconformer --model {IC} --run tts_sent_indicconformer > /dev/null")
sh(f"{EVAL_PY} task2_asr/eval_tts_sentences.py --backend whisper --model bengaliAI/tugstugi_bengaliai-asr_whisper-medium --run tts_sent_tugstugi > /dev/null")

rows = []
for run in ["tts_sent_indicconformer", "tts_sent_tugstugi"]:
    s = json.load(open(f"{REPO}/results/task2/{run}/summary.json", encoding="utf-8"))
    rows.append({"model": run.replace("tts_sent_", ""), "WER%": s["ALL"]["WER%"], "CER%": s["ALL"]["CER%"],
                 "MED CER%": s["MED"]["CER%"], "SHORT CER%": s["SHORT"]["CER%"], "plain CER%": s["untagged"]["CER%"],
                 "term acc%": s["ALL"]["term_acc%"], "term acc, spaces ignored%": s["ALL"]["term_acc_ignoring_spaces%"]})
pd.DataFrame(rows).set_index("model")

## 6. Download the outputs

Generated audio, labels and results as one zip.

In [ ]:
paths = [f"{DATA}/tts_sentences", f"{DATA}/tts_consultation", f"{REPO}/data/tts", f"{REPO}/data/tts_consultation",
         *[f"{REPO}/results/{r}" for r in ["task2/tts_sent_indicconformer", "task2/tts_sent_tugstugi", "task3/tts_baseline", "task3/tts_final"]]]
for p in paths:
    shutil.copytree(p, f"/content/tts_outputs/{os.path.basename(os.path.dirname(p))}_{os.path.basename(p)}", dirs_exist_ok=True)
shutil.make_archive("/content/tts_outputs", "zip", "/content/tts_outputs")
from google.colab import files
files.download("/content/tts_outputs.zip")